# 04 Transfer Learning for Object Detection

## 📚 Learning Objectives

By completing this notebook (~20 min), you will:
- Understand how **object detection** uses a **pre-trained backbone** + **detection head**
- Use a **pre-trained CNN** as a feature extractor and add a **simple classification head** on top (simplified “detection” setup)
- See why we use transfer learning for detection instead of training from scratch

---

## 🌍 Real life

**Where is this used?** Object detection (localize + classify) is used in **autonomous driving**, **surveillance**, and **retail** (shelf monitoring).

**In this notebook we use** a **pre-trained backbone** (e.g. MobileNetV2) to extract features, then add a **head** for classification. We use **transfer learning for detection** (instead of training a detector from scratch) **because** the backbone already learned good visual features; we only train the head (or fine-tune last layers) with less data.

**📌 Covers slide(s):** **14**, **15** — Object Detection (Faster R-CNN, SSD, YOLO). *Do this notebook after those slides.*

---

**Before starting:** Run the imports cell below. Full object detection (bounding boxes) uses libraries like TensorFlow Object Detection API; here we show the **backbone + head** idea in ~20 min.

⏱ **Runtime:** This notebook may take 10–40 minutes on GPU (depending on backbone and epochs). Use a smaller subset or fewer epochs if needed (see unit README).

## Theory (short)

- **Object detection:** Find **where** objects are (bounding boxes) and **what** they are (class).
- **Typical pipeline:** Pre-trained **backbone** (e.g. ResNet, MobileNet) → **neck** (e.g. FPN) → **detection head** (boxes + classes). YOLO, SSD, Faster R-CNN follow this idea.
- **Transfer learning:** Backbone is pre-trained on ImageNet; we freeze or fine-tune it and train the detection head on our dataset.
- **We use a pre-trained backbone** instead of training from scratch so we need less data and time; the head learns “where” and “what” on top of good features.

## 📥 Inputs & 📤 Outputs

**Inputs:** TensorFlow/Keras, NumPy. We use **MNIST resized to 96×96 RGB** (as in 05_transfer_learning_cnns) so the notebook runs without an object-detection dataset.

**Dataset:** Real — MNIST (resized to 96×96 RGB for backbone demo).

**Outputs:** Model summary (backbone + head), training loss/accuracy for 2 epochs, and test accuracy. (Full detection would output bounding boxes; here we do **image-level classification** to show the backbone+head pattern.)

## Step 1: Imports and load pre-trained backbone (we use MobileNetV2 as backbone instead of training from scratch)

In [1]:
import numpy as np

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    HAS_TF = False

if HAS_TF:
    backbone = keras.applications.MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights="imagenet")
    backbone.trainable = False
    print("Backbone (frozen) params:", backbone.count_params())
else:
    print("Install TensorFlow: pip install tensorflow")

Backbone (frozen) params: 2257984


## Step 2: Add classification head (in full detection we would add a head that outputs boxes + classes)

In [2]:
if HAS_TF:
    inp = keras.Input(shape=(96, 96, 3))
    x = backbone(inp)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inp, x)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("Model: backbone + global pool + Dense(10). For real detection, head would output boxes + classes.")

Model: backbone + global pool + Dense(10). For real detection, head would output boxes + classes.


## Step 3: Prepare data (MNIST as 96×96 RGB) and train head (2 epochs)

In [3]:
if HAS_TF:
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    x_train = tf.image.resize(x_train[..., np.newaxis], (96, 96))
    x_test = tf.image.resize(x_test[..., np.newaxis], (96, 96))
    x_train = tf.repeat(x_train, 3, axis=-1).numpy().astype(np.float32) / 255.0
    x_test = tf.repeat(x_test, 3, axis=-1).numpy().astype(np.float32) / 255.0
    x_train, y_train = x_train[:5000], y_train[:5000]
    history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=2, batch_size=64, verbose=1)
    _, acc = model.evaluate(x_test, y_test, verbose=0)
    print("Test accuracy: %.4f" % acc)

Epoch 1/2


 1/79 ━━━━━━━━━━━━━━━━━━━━ 1:41 1s/step - accuracy: 0.0938 - loss: 2.7851

 2/79 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - accuracy: 0.0820 - loss: 2.7881

 4/79 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.1130 - loss: 2.6534

 5/79 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.1241 - loss: 2.6059

...

76/79 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9162 - loss: 0.3148

77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9163 - loss: 0.3144

78/79 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9164 - loss: 0.3139

79/79 ━━━━━━━━━━━━━━━━━━━━ 12s 159ms/step - accuracy: 0.9254 - loss: 0.2792 - val_accuracy: 0.9296 - val_loss: 0.2542


Test accuracy: 0.9296


## 🌍 Real-World Worked Example — Fine-Tune ResNet on Custom Categories

**Industry context:**
- Google Photos uses transfer learning to classify your personal photos  
- Hospitals fine-tune ImageNet models on their X-ray datasets with <1000 images
- E-commerce platforms fine-tune ResNet to identify product defects

We fine-tune a **pretrained ResNet-18** (ImageNet weights) on a small binary classification task.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

# ── Use CIFAR-10 classes 0 (airplane) vs 1 (automobile) as our 'custom' data
transform = T.Compose([
    T.Resize(64), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])  # ImageNet stats
])
full = torchvision.datasets.CIFAR10('/tmp/cifar10', train=True, download=True, transform=transform)
# Keep only classes 0 and 1
idx = [i for i,(x,y) in enumerate(full) if y in (0,1)][:400]
subset = Subset(full, idx)
train_size = int(0.8*len(subset))
train_ds, val_ds = torch.utils.data.random_split(subset, [train_size, len(subset)-train_size])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=32)

# ── Load pretrained ResNet-18, replace final layer ──────────────────────────
model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for p in model.parameters(): p.requires_grad = False        # Freeze backbone
model.fc = nn.Linear(model.fc.in_features, 2)               # Only train head

opt     = optim.Adam(model.fc.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train(); total_loss=0
    for X,y in train_dl:
        y_bin = (y % 2)  # remap to 0/1
        loss = loss_fn(model(X), y_bin)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
    model.eval(); correct=0; total=0
    with torch.no_grad():
        for X,y in val_dl:
            y_bin = (y%2)
            correct += (model(X).argmax(1)==y_bin).sum().item(); total+=len(y_bin)
    print(f"Epoch {epoch+1}/5 — loss: {total_loss/len(train_dl):.3f} | val acc: {correct/total*100:.1f}%")

print("\n✅ With only 400 images and 5 epochs, transfer learning gives strong results.")
print("A model trained from scratch would need 100x more data for similar performance.")

## 🧩 Mini-exercise

**Try it:** Change the number of units in the classification head (e.g. 64 → 128) and retrain for 1 epoch. Does validation accuracy change? Or try a different base model (e.g. ResNet50) if available and compare training time.

---

## ✅ Summary

**What you did:** Used a pre-trained backbone (MobileNetV2) + a classification head, trained only the head on MNIST (resized), and saw how transfer learning applies to a detection-style setup.

**In real life you'd also:** Use a real detection dataset (e.g. COCO), add a head that outputs bounding boxes and classes, and use TensorFlow Object Detection API or similar.

**The main idea:** Object detection often uses a pre-trained backbone + a detection head; transfer learning lets us train the head (and optionally fine-tune the backbone) with limited data.

**Next:** `05_transfer_learning_cnns` does transfer learning for classification; for full detection pipelines see TensorFlow Object Detection API.

## 📚 References & Further Reading

**Papers:**
- Tan et al. (2019) — [EfficientNet](https://arxiv.org/abs/1905.11946)
- Dosovitskiy et al. (2020) — [ViT: Vision Transformer](https://arxiv.org/abs/2010.11929)
- He et al. (2016) — [ResNet](https://arxiv.org/abs/1512.03385)

**Practical Guide:** [torchvision Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)

**State-of-the-Art:** In 2025, fine-tuning a pretrained ViT-L on 100 medical images achieves radiologist-level performance.